# NL2Shell — Fine-tune Qwen3-0.6B on NL2Bash

**By Arya Teja | LeCoder | CloudAGI**

Fine-tunes Qwen3-0.6B with QLoRA on NL2Bash dataset for natural language → shell command translation.
Exports GGUF for Raspberry Pi (Q4) and Mac (Q8). Pushes to HuggingFace.

**Runtime:** A100 GPU + High RAM | **Time:** ~45 min | **Cost:** ~10 compute units

## Instructions
1. Set runtime: Runtime > Change runtime type > A100 + High RAM
2. Run Cell 1, then **restart runtime** (Runtime > Restart runtime)
3. Run cells 2-10 in order
4. Set your HF_TOKEN in Cell 2 or Colab Secrets
5. Close laptop after training starts — Colab Pro keeps running

In [ ]:
# =================================================================
# CELL 1: Install Dependencies
# After this cell: Runtime > Restart runtime, then run cells 2-10
# =================================================================

!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -q datasets trl peft accelerate bitsandbytes transformers huggingface_hub

print("\n✅ Install complete. NOW: Runtime > Restart runtime")
print("Then run cells 2-10 WITHOUT re-running this cell.")

In [ ]:
# =================================================================
# CELL 2: Config — EDIT THESE
# =================================================================

import torch, os

# ---- EDIT THESE ----
HF_TOKEN     = ""   # paste your HF write token, OR use Colab Secrets
HF_USERNAME  = "AryaYT"
HF_REPO      = "nl2shell-0.6b"

# ---- Training config ----
MODEL_ID        = "unsloth/Qwen3-0.6B-bnb-4bit"
MAX_SEQ_LENGTH  = 512
LORA_RANK       = 16
BATCH_SIZE      = 8
GRAD_ACCUM      = 4    # effective batch = 32
EPOCHS          = 3
LR              = 2e-4
SAVE_STEPS      = 100
CHECKPOINT_DIR  = "/content/drive/MyDrive/nl2shell-checkpoints"

SYSTEM_PROMPT = "You are an expert shell programmer. Given a natural language description, output the exact shell command. Output only the command, no explanation."

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")
print(f"HF repo: {HF_USERNAME}/{HF_REPO}")

In [ ]:
# =================================================================
# CELL 3: Mount Drive + HF Auth
# =================================================================

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_DIR}")

if not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab Secrets")
    except:
        print("⚠️ No HF token. Add via sidebar > key icon > HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ HuggingFace authenticated")

In [ ]:
# =================================================================
# CELL 4: Load Model + QLoRA
# =================================================================

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_RANK * 2,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()

In [ ]:
# =================================================================
# CELL 5: Load NL2Bash + Format Dataset
# =================================================================

from datasets import load_dataset, concatenate_datasets, Dataset

dataset = load_dataset("jiacheng-ye/nl2bash")
print(dataset)
print(f"\nSample: {dataset['train'][0]}")

def format_chatml(example):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['nl']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['bash']}<|im_end|>"
    )
    return {"text": text}

# Format main dataset
train_data = dataset["train"].map(format_chatml, remove_columns=["nl", "bash"])
val_data   = dataset["validation"].map(format_chatml, remove_columns=["nl", "bash"])

# Add macOS-specific pairs
macos_pairs = [
    ("list all installed homebrew packages", "brew list"),
    ("update homebrew and upgrade all packages", "brew update && brew upgrade"),
    ("show disk usage of current directory", "du -sh ."),
    ("find all Python files modified in the last 24 hours", "find . -name '*.py' -mtime -1"),
    ("show all running Docker containers", "docker ps"),
    ("kill the process using port 3000", "lsof -ti:3000 | xargs kill -9"),
    ("create a new git branch called feature-auth", "git checkout -b feature-auth"),
    ("show git log as one-line summaries", "git log --oneline -20"),
    ("compress the src directory into a tar.gz", "tar -czf src.tar.gz src/"),
    ("show system memory usage on macOS", "vm_stat | head -10"),
    ("list all open network connections", "lsof -i -P -n | head -20"),
    ("recursively find files larger than 100MB", "find / -size +100M -type f 2>/dev/null"),
    ("restart the DNS cache on macOS", "sudo dscacheutil -flushcache && sudo killall -HUP mDNSResponder"),
    ("show all environment variables containing PATH", "env | grep PATH"),
    ("count lines of code in all TypeScript files", "find . -name '*.ts' | xargs wc -l"),
    ("check if port 8080 is in use", "lsof -i :8080"),
    ("show the top 10 largest files in current directory", "ls -lhS | head -10"),
    ("create an SSH tunnel from local 8080 to remote 80", "ssh -L 8080:localhost:80 user@host"),
    ("install a package globally with npm", "npm install -g package-name"),
    ("run a Python HTTP server on port 8000", "python3 -m http.server 8000"),
    ("show all cron jobs for current user", "crontab -l"),
    ("search for a string recursively in all files", "grep -r 'search_string' ."),
    ("generate a random 32-character password", "openssl rand -base64 32"),
    ("download a file with curl and save it", "curl -LO https://example.com/file.tar.gz"),
    ("show the size of each subdirectory", "du -sh */ | sort -rh"),
    ("list all git branches sorted by last commit date", "git branch --sort=-committerdate"),
    ("find and delete all node_modules directories", "find . -name 'node_modules' -type d -prune -exec rm -rf {} +"),
    ("show which process is using the most CPU", "ps aux --sort=-%cpu | head -5"),
    ("convert a video to mp4 using ffmpeg", "ffmpeg -i input.mov -c:v libx264 -c:a aac output.mp4"),
    ("run a command in the background and disown it", "nohup command &>/dev/null & disown"),
]

macos_data = Dataset.from_dict({
    "text": [
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{nl}<|im_end|>\n"
        f"<|im_start|>assistant\n{cmd}<|im_end|>"
        for nl, cmd in macos_pairs
    ]
})

train_data = concatenate_datasets([train_data, macos_data]).shuffle(seed=42)
print(f"\nTotal train: {len(train_data)} | Val: {len(val_data)}")
print(f"\nSample:\n{train_data[0]['text']}")

In [ ]:
# =================================================================
# CELL 6: Start Checkpoint Backup Thread (run BEFORE training)
# =================================================================

import threading, time, shutil
from datetime import datetime

def backup_loop(interval=900):
    drive_backup = "/content/drive/MyDrive/nl2shell-backup"
    os.makedirs(drive_backup, exist_ok=True)
    while True:
        time.sleep(interval)
        try:
            ckpts = sorted([d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")])
            if ckpts:
                src = os.path.join(CHECKPOINT_DIR, ckpts[-1])
                ts = datetime.now().strftime("%H%M%S")
                shutil.copytree(src, f"{drive_backup}/{ckpts[-1]}-{ts}", dirs_exist_ok=True)
                print(f"[{ts}] Backed up {ckpts[-1]}")
        except Exception as e:
            print(f"Backup error: {e}")

threading.Thread(target=backup_loop, daemon=True).start()
print("✅ Checkpoint backup thread running (every 15 min -> Drive)")

In [ ]:
# =================================================================
# CELL 7: Train
# ~30-45 min on A100 for 3 epochs on 8K examples
# =================================================================

from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_data,
    eval_dataset       = val_data,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,
    args = TrainingArguments(
        output_dir                  = CHECKPOINT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LR,
        warmup_ratio                = 0.03,
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,
        bf16                        = True,
        logging_steps               = 20,
        eval_strategy               = "steps",
        eval_steps                  = 100,
        save_strategy               = "steps",
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        report_to                   = "none",
        dataloader_num_workers      = 2,
    ),
)

print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Checkpoints -> {CHECKPOINT_DIR}")
print("\n🚀 Training...")

stats = trainer.train()

print(f"\n✅ Done! Loss: {stats.training_loss:.4f}")
print(f"Time: {stats.metrics['train_runtime'] / 60:.1f} min")

In [ ]:
# =================================================================
# CELL 8: Quick Eval
# =================================================================

FastLanguageModel.for_inference(model)

tests = [
    "list all files in the current directory",
    "find all Python files larger than 1MB",
    "kill all processes named python",
    "compress the logs directory into a tar.gz archive",
    "check which process is using port 8080",
    "show git log as one-line summaries",
    "update homebrew and upgrade all packages",
]

print("=" * 60)
for nl in tests:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{nl}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, temperature=0.1,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    cmd = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()
    print(f"NL:  {nl}")
    print(f"CMD: {cmd}\n")

In [ ]:
# =================================================================
# CELL 9: Export GGUF + Push to HuggingFace
# =================================================================

from huggingface_hub import HfApi, create_repo
import shutil

FULL_REPO = f"{HF_USERNAME}/{HF_REPO}"
api = HfApi()

# Save LoRA
print("Saving LoRA adapter...")
model.save_pretrained("/content/nl2shell-lora")
tokenizer.save_pretrained("/content/nl2shell-lora")

# Merge + save 16bit
print("Merging LoRA into base...")
model.save_pretrained_merged("/content/nl2shell-merged", tokenizer, save_method="merged_16bit")

# GGUF exports
for quant in ["q4_k_m", "q8_0"]:
    print(f"Exporting GGUF {quant}...")
    try:
        model.save_pretrained_gguf(f"/content/nl2shell-{quant}", tokenizer,
                                   quantization_method=quant)
        print(f"  ✅ {quant}")
    except Exception as e:
        print(f"  ⚠️ {quant} failed: {e}")

# Copy GGUF to Drive
for quant in ["q4_k_m", "q8_0"]:
    src = f"/content/nl2shell-{quant}"
    if os.path.exists(src):
        shutil.copytree(src, f"/content/drive/MyDrive/nl2shell-gguf/{quant}", dirs_exist_ok=True)
print("GGUF backed up to Drive")

# Push merged model
print(f"\nPushing to {FULL_REPO}...")
create_repo(FULL_REPO, private=False, exist_ok=True)
api.upload_folder(folder_path="/content/nl2shell-merged", repo_id=FULL_REPO,
                  repo_type="model", commit_message="NL2Shell 0.6B — QLoRA fine-tuned on NL2Bash")

# Push GGUF files
for quant in ["q4_k_m", "q8_0"]:
    path = f"/content/nl2shell-{quant}"
    if os.path.exists(path):
        for f in os.listdir(path):
            if f.endswith(".gguf"):
                api.upload_file(path_or_fileobj=os.path.join(path, f),
                               path_in_repo=f"gguf/{f}", repo_id=FULL_REPO)
                print(f"  Uploaded gguf/{f}")

print(f"\n🎉 Model live: https://huggingface.co/{FULL_REPO}")

In [ ]:
# =================================================================
# CELL 10: Model Card
# =================================================================

MODEL_CARD = """---
license: mit
base_model: Qwen/Qwen3-0.6B
tags:
  - nl2bash
  - shell
  - terminal
  - command-line
  - qlora
  - lecoder
  - cloudagi
datasets:
  - jiacheng-ye/nl2bash
language:
  - en
pipeline_tag: text-generation
---

# NL2Shell 0.6B

Ultra-lightweight model for converting natural language to Unix/macOS shell commands.
Fine-tuned from Qwen3-0.6B using QLoRA on NL2Bash + macOS-specific command pairs.

## Why

Every terminal interaction with a coding agent sends your shell commands to cloud providers.
NL2Shell runs locally — on a MacBook, Raspberry Pi, or any device with 400MB free.
Your shell history stays on your machine.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("AryaYT/nl2shell-0.6b")
tokenizer = AutoTokenizer.from_pretrained("AryaYT/nl2shell-0.6b")
```

### GGUF for Ollama / llama.cpp
- `gguf/nl2shell-q4_k_m.gguf` — ~400MB, Raspberry Pi compatible
- `gguf/nl2shell-q8_0.gguf` — ~650MB, higher quality for Mac/desktop

## Training
- Base: Qwen/Qwen3-0.6B
- Method: QLoRA (rank 16, all linear layers)
- Data: NL2Bash (8090 examples) + 30 macOS synthetic pairs
- Hardware: Google Colab Pro A100
- Framework: Unsloth

## Built by
[Arya Teja](https://aryateja.com) — Agentic Engineer, SF
[LeCoder](https://lecoder.lesearch.ai) | [CloudAGI](https://cloudagi.ai)
"""

api.upload_file(path_or_fileobj=MODEL_CARD.encode(),
                path_in_repo="README.md", repo_id=FULL_REPO)
print(f"✅ Model card uploaded")
print(f"\n🎉 ALL DONE: https://huggingface.co/{FULL_REPO}")
print("\nNext: Download GGUF and test with Ollama:")
print(f"  ollama run hf.co/{FULL_REPO}")